# Road PCI Intelligence — Stage 3D: Exhaustive RDD Box Statistics

This notebook exhaustively analyzes all 25,599 RDD label files using 16 threads:
- **Classes**: 0: longitudinal, 1: longitudinal wide, 2: transverse, 3: transverse wide, 4: alligator, 5: alligator sunken, 6: pothole, 7: pothole deep.
- **Splits**: train, val, test, and overall.
- **Metrics**: Boxes per class, images with zero boxes, boxes-per-image histogram, box area fraction median & 90th percentile.


In [ ]:
# ==============================================================================
# 1. ENVIRONMENT & REPOSITORY SETUP
# ==============================================================================
import os
import sys
import time
import subprocess
from pathlib import Path

repo_dir = Path('/kaggle/working/repo')
if not repo_dir.exists():
    print('Cloning repository...')
    subprocess.run(['git', 'clone', 'https://github.com/skilloutofbounds/road-pci-intelligence.git', str(repo_dir)], check=True)
else:
    print('Pulling latest changes...')
    subprocess.run(['git', '-C', str(repo_dir), 'pull'], check=True)

# Locate RDD dataset
candidates = [
    Path('/kaggle/input/datasets/juusos/rdd2022es'),
    Path('/kaggle/input/rdd2022es'),
    Path('data/rdd2022es')
]
dataset_root = None
for c in candidates:
    if c.exists() and c.is_dir():
        dataset_root = c.resolve()
        break

if dataset_root is None:
    for d in Path('/kaggle/input').rglob('rdd2022es'):
        if d.is_dir():
            dataset_root = d.resolve()
            break

assert dataset_root is not None, 'Could not find RDD dataset root!'
print(f'RDD Dataset Root: {dataset_root}')


In [ ]:
# ==============================================================================
# 2. EXHAUSTIVE MULTI-THREADED AUDIT (16 THREADS)
# ==============================================================================
import json
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

manifest_p = repo_dir / 'manifests/rdd2022es_manifest.csv'
assert manifest_p.exists(), f'Missing manifest: {manifest_p}'
df_rdd = pd.read_csv(manifest_p)
print(f'Loaded manifest with {len(df_rdd):,} records.')

annotated_dir = dataset_root / 'combined_annotatedv2'
if not annotated_dir.exists():
    dirs = list(dataset_root.rglob('combined_annotatedv2'))
    annotated_dir = dirs[0] if dirs else dataset_root

all_txt_files = list(annotated_dir.glob('*.txt'))
print(f'Total .txt label files discovered in {annotated_dir.name}: {len(all_txt_files):,}')

RDD_CLASSES = {
    0: 'longitudinal',
    1: 'longitudinal wide',
    2: 'transverse',
    3: 'transverse wide',
    4: 'alligator',
    5: 'alligator sunken',
    6: 'pothole',
    7: 'pothole deep'
}

work_items = []
for _, row in df_rdd.iterrows():
    txt_rel = row['mask_path_or_label_path'] if pd.notna(row['mask_path_or_label_path']) else ''
    work_items.append((txt_rel, row['split'], row['group_id'], row['image_path']))

def parse_item(item):
    txt_rel, split, gid, img_rel = item
    txt_p = dataset_root / txt_rel if txt_rel else None
    boxes = []
    if txt_p and txt_p.exists():
        try:
            with open(txt_p, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cid = int(float(parts[0]))
                        bw = float(parts[3])
                        bh = float(parts[4])
                        area_frac = bw * bh
                        boxes.append((cid, area_frac))
        except Exception:
            pass
    return {
        'split': split,
        'gid': gid,
        'img_rel': img_rel,
        'n_boxes': len(boxes),
        'boxes': boxes
    }

print('Parsing all label files in parallel using 16 threads...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as pool:
    results = list(pool.map(parse_item, work_items))
t_elapsed = time.time() - t0
print(f'Parsed {len(results):,} label files in {t_elapsed:.2f}s ({len(results)/t_elapsed:.1f} files/s).\n')


In [ ]:
# ==============================================================================
# 3. AGGREGATE & REPORT METRICS PER SPLIT (TRAIN / VAL / TEST / OVERALL)
# ==============================================================================
output_summary = {}

splits = ['train', 'val', 'test', 'overall']

for sp in splits:
    if sp == 'overall':
        sp_records = results
    else:
        sp_records = [r for r in results if r['split'] == sp]
        
    n_imgs = len(sp_records)
    n_zero = sum(1 for r in sp_records if r['n_boxes'] == 0)
    zero_pct = (n_zero / n_imgs) * 100 if n_imgs > 0 else 0.0
    
    box_counts = Counter()
    box_areas = defaultdict(list)
    boxes_per_img = [r['n_boxes'] for r in sp_records]
    
    for r in sp_records:
        for cid, area_frac in r['boxes']:
            box_counts[cid] += 1
            box_areas[cid].append(area_frac)
            
    tot_boxes = sum(box_counts.values())
    mean_bpi = np.mean(boxes_per_img) if boxes_per_img else 0.0
    med_bpi = np.median(boxes_per_img) if boxes_per_img else 0.0
    max_bpi = max(boxes_per_img) if boxes_per_img else 0
    
    # Histogram
    hist = Counter(boxes_per_img)
    
    print('=' * 95)
    print(f'RDD2022ES EXHAUSTIVE BOX AUDIT: SPLIT = {sp.upper()} (N = {n_imgs:,} IMAGES)')
    print('=' * 95)
    print(f'  • Total Images:               {n_imgs:,}')
    print(f'  • Images with Zero Boxes:     {n_zero:,} ({zero_pct:.2f}%)')
    print(f'  • Images with >= 1 Box:       {n_imgs - n_zero:,} ({100 - zero_pct:.2f}%)')
    print(f'  • Total Bounding Boxes:       {tot_boxes:,}')
    print(f'  • Mean Boxes per Image:       {mean_bpi:.2f} (Median: {med_bpi:.1f}, Max: {max_bpi})\n')
    
    print('  [Boxes-Per-Image Histogram Distribution]')
    for k in range(10):
        cnt = hist[k]
        pct = (cnt / n_imgs) * 100 if n_imgs > 0 else 0.0
        print(f'    - {k:2d} boxes: {cnt:6,d} images ({pct:5.2f}%)')
    gte_10 = sum(cnt for num_b, cnt in hist.items() if num_b >= 10)
    gte_10_pct = (gte_10 / n_imgs) * 100 if n_imgs > 0 else 0.0
    print(f'    - >=10 boxes: {gte_10:6,d} images ({gte_10_pct:5.2f}%)\n')
    
    print(f'  [Per-Class Box Counts & Area Fractions (% of Image Area)]')
    print(f'  {"Class ID":<8} | {"Class Name":<22} | {"Total Boxes":<12} | {"Median Box Area":<16} | {"90th Pct Box Area":<16}')
    print('  ' + '-' * 86)
    
    split_class_stats = {}
    for cid in range(8):
        cname = RDD_CLASSES[cid]
        c_boxes = box_counts[cid]
        areas = box_areas[cid]
        med_area = float(np.median(areas) * 100) if areas else 0.0
        p90_area = float(np.percentile(areas, 90) * 100) if areas else 0.0
        
        split_class_stats[cid] = {
            'class_name': cname,
            'total_boxes': c_boxes,
            'median_area_pct': med_area,
            'p90_area_pct': p90_area
        }
        print(f'  {cid:<8d} | {cname:<22} | {c_boxes:<12,d} | {med_area:13.3f}% | {p90_area:15.3f}%')
    print('=' * 95 + '\n')
    
    output_summary[sp] = {
        'total_images': n_imgs,
        'zero_box_images': n_zero,
        'zero_box_pct': zero_pct,
        'total_boxes': tot_boxes,
        'mean_boxes_per_image': float(mean_bpi),
        'median_boxes_per_image': float(med_bpi),
        'max_boxes_per_image': int(max_bpi),
        'histogram': {int(k): int(v) for k, v in hist.items()},
        'class_stats': split_class_stats
    }

# Save JSON
out_json = Path('/kaggle/working/rdd_box_stats.json')
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(output_summary, f, indent=2)
print(f'[OK] Saved exhaustive box stats to {out_json}')
